# Traffic Sign Label-Flipping Poisoning Assessment: Solution

This notebook implements a simple data poisoning assessment for a traffic sign classifier. The poison changes labels only: a subset of `Stop` sign training samples are relabeled as `Yield`, then the classifier is retrained and evaluated on clean validation data.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

os.environ["ART_DATA_PATH"] = str(ROOT / "art_data")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")

import numpy as np
import torch

from traffic_sign_poisoning_utils import (
    SELECTED_CLASS_NAMES,
    SOURCE_CLASS_ID,
    SOURCE_CLASS_NAME,
    TARGET_CLASS_ID,
    TARGET_CLASS_NAME,
    build_traffic_sign_cnn,
    class_accuracy_table,
    create_label_flip_poisoned_training_set,
    evaluate_clean,
    plot_label_flip_examples,
    prepare_gtsrb_subsets,
    save_checkpoint,
    targeted_misclassification_rate,
    train_model,
    write_results_table,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = ROOT / "data" / "generated"
DOWNLOAD_DIR = ROOT / "data" / "gtsrb"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(7)
np.random.seed(7)

print(f"Device: {DEVICE}")
print(SELECTED_CLASS_NAMES)

## 1. Prepare the Compact GTSRB Subset

In [ ]:
prepare_gtsrb_subsets(DATA_DIR, DOWNLOAD_DIR, train_per_class=80, val_per_class=30)
train_data = np.load(DATA_DIR / "traffic_sign_train_clean.npz")
val_data = np.load(DATA_DIR / "traffic_sign_val_clean.npz")

train_images, train_labels = train_data["images"], train_data["labels"]
val_images, val_labels = val_data["images"], val_data["labels"]

print(f"Training images: {train_images.shape}")
print(f"Validation images: {val_images.shape}")
print(f"Source class: {SOURCE_CLASS_NAME} -> target class: {TARGET_CLASS_NAME}")

## 2. Train and Evaluate the Clean Baseline

In [ ]:
clean_model = build_traffic_sign_cnn()
clean_model = train_model(clean_model, train_images, train_labels, device=DEVICE, epochs=4)
save_checkpoint(clean_model, MODEL_DIR / "clean_traffic_sign_cnn.pt")

clean_eval = evaluate_clean(clean_model, val_images, val_labels, device=DEVICE)
clean_class_rows = class_accuracy_table(clean_eval["predictions"], val_labels)
clean_source_accuracy = [row for row in clean_class_rows if row["class_id"] == SOURCE_CLASS_ID][0]["accuracy"]
clean_targeted_rate = targeted_misclassification_rate(clean_eval["predictions"], val_labels, SOURCE_CLASS_ID, TARGET_CLASS_ID)

print(f"Clean baseline accuracy: {clean_eval['accuracy']:.3f}")
print(f"Clean source-class accuracy: {clean_source_accuracy:.3f}")
print(f"Clean targeted misclassification rate: {clean_targeted_rate:.3f}")
clean_class_rows

## 3. Create a Label-Flipped Training Set

This attack flips training labels from `Stop` to `Yield`. No trigger is added and the image pixels remain unchanged.

In [ ]:
POISON_FRACTION = 0.75

poisoned_images, poisoned_labels, flipped_indices = create_label_flip_poisoned_training_set(
    train_images,
    train_labels,
    source_class=SOURCE_CLASS_ID,
    target_class=TARGET_CLASS_ID,
    poison_fraction=POISON_FRACTION,
)

print(f"Flipped labels: {len(flipped_indices)}")
print(f"Original source-class training samples: {np.sum(train_labels == SOURCE_CLASS_ID)}")
print(f"Poisoned labels now marked as target: {np.sum(poisoned_labels[flipped_indices] == TARGET_CLASS_ID)}")

In [ ]:
plot_label_flip_examples(
    train_images,
    train_labels,
    poisoned_labels,
    flipped_indices,
    RESULTS_DIR / "label_flip_examples.png",
)

## 4. Retrain on Poisoned Labels

In [ ]:
poisoned_model = build_traffic_sign_cnn()
poisoned_model = train_model(poisoned_model, poisoned_images, poisoned_labels, device=DEVICE, epochs=4)
save_checkpoint(poisoned_model, MODEL_DIR / "label_flipped_traffic_sign_cnn.pt")

poisoned_eval = evaluate_clean(poisoned_model, val_images, val_labels, device=DEVICE)
poisoned_class_rows = class_accuracy_table(poisoned_eval["predictions"], val_labels)
poisoned_source_accuracy = [row for row in poisoned_class_rows if row["class_id"] == SOURCE_CLASS_ID][0]["accuracy"]
poisoned_targeted_rate = targeted_misclassification_rate(poisoned_eval["predictions"], val_labels, SOURCE_CLASS_ID, TARGET_CLASS_ID)

print(f"Poisoned model clean accuracy: {poisoned_eval['accuracy']:.3f}")
print(f"Poisoned model source-class accuracy: {poisoned_source_accuracy:.3f}")
print(f"Poisoned model targeted misclassification rate: {poisoned_targeted_rate:.3f}")
poisoned_class_rows

## 5. Compare Aggregate Accuracy and Targeted Behavior

In [ ]:
rows = [
    {
        "model": "clean_baseline",
        "clean_accuracy": round(clean_eval["accuracy"], 4),
        "source_class_accuracy": round(clean_source_accuracy, 4),
        "targeted_misclassification_rate": round(clean_targeted_rate, 4),
        "mean_confidence": round(clean_eval["mean_confidence"], 4),
        "notes": "trained_on_clean_labels",
    },
    {
        "model": "label_flipped_model",
        "clean_accuracy": round(poisoned_eval["accuracy"], 4),
        "source_class_accuracy": round(poisoned_source_accuracy, 4),
        "targeted_misclassification_rate": round(poisoned_targeted_rate, 4),
        "mean_confidence": round(poisoned_eval["mean_confidence"], 4),
        "notes": f"{POISON_FRACTION:.0%}_of_source_labels_flipped_to_target",
    },
]

write_results_table(rows, RESULTS_DIR / "label_flip_metrics.csv")
rows

## 6. Operational Risk Summary

A label-flipping attack can be operationally risky even when images look normal, because the training labels silently teach the model the wrong relationship between safety-critical classes. Aggregate clean accuracy alone may hide the issue, especially when the affected class is a small share of the validation set. For transportation systems, the risk should be assessed with class-specific validation, targeted confusion analysis, label provenance checks, and review gates for high-impact classes such as stop signs.